# S4V8: solar filament instance segmentation, end to end
This notebook runs the whole pipeline: preprocessing, training (optional), inference with post-processing, evaluation, and a look at the predictions.

**By default nothing is trained.** The notebook rebuilds the stored weights from `weight_parts/` and predicts with them. Set `TRAIN = True` to retrain all 12 models from the public pretrained weights instead (about 40 GPU hours, see section 3). Inference then uses the retrained models; the evaluation uses them only with `EVALUATE = True` (otherwise it shows the stored evaluation of the delivered weights, labelled as such).

Settings:
- `TEST_IMAGES`: folder of test H-alpha JPEGs (inference uses only these).
- `TRAIN_DATA`: competition training folder (`train_images/` and the train annotation JSON). It is needed only for `TRAIN` and `EVALUATE`.
- `LIMIT`: number of test images to predict (0 = all; about 14 s per image on an RTX 3080 Ti).

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get('S4V8_ROOT', '.')).resolve()
TEST_IMAGES = (ROOT / os.environ.get('S4V8_IMAGES', 'test_images')).resolve()     # relative paths are taken from the repository root
TRAIN_DATA = (ROOT / os.environ.get('S4V8_TRAIN_DATA', 'train_data')).resolve()
LIMIT = int(os.environ.get('S4V8_LIMIT', '3'))
TRAIN = os.environ.get('S4V8_TRAIN', '0') == '1'        # retrain all models (section 3); default: load the stored weights
EVALUATE = os.environ.get('S4V8_EVALUATE', '0') == '1'  # out-of-fold evaluation on the training images (section 5, about 1 GPU hour)
OUT = ROOT / 'submission_demo.csv'
ENV = dict(os.environ, PYTHONUTF8='1')
assert (ROOT / 'params.json').exists(), 'run this notebook from the repository root'
assert TEST_IMAGES.is_dir(), f'set TEST_IMAGES: {TEST_IMAGES} not found'
sys.path.insert(0, str(ROOT / 'code'))

def run(args):
    print('$', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, check=True, env=ENV)

## 1. Rebuild the stored weights
The pieces in `weight_parts/` are joined into `weights/`. Every piece and every rebuilt file is checked against SHA-256.

In [ ]:
run([ROOT / 'assemble_weights.py'])

## 2. Preprocessing
Every model reads the H-alpha image as greyscale at 2048 x 2048 and resizes it to 1536 x 1536.
- **Mask R-CNN `ep24`**: the grey image repeated in 3 channels.
- **Mask R-CNN `e034`**: three physics channels computed from the image alone: grey; limb darkening flattened by a radial median profile of the disk; a multi-scale black top-hat that highlights thin dark structures.
- **Dense U-Nets `D1`, `D2`**: the grey image as one channel (PIL box resize to 1536), standardised per image (zero mean, unit variance).

At test time each model also sees 8 flip and transpose views of the image. Its predictions are mapped back to the original orientation.

In [ ]:
import cv2, numpy as np
from IPython.display import Image, display
import e006_maskrcnn_refiner as e6, e056_infer
example = sorted(p for p in TEST_IMAGES.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg'))[0]
grey = e6.resize_input(e6.load_gray(example), 1536)            # float in [0, 1], 1536 x 1536
phys = e6.phys_channels(grey)                                   # 3 x 1536 x 1536
d = e056_infer.load_input(example, 1536).astype(np.float32) / 255   # the dense models' own loader
dense_in = (d - d.mean()) / (d.std(ddof=1) + 1e-6)                  # as torch .std() at inference
tiles = [grey, phys[1], phys[2], (dense_in - dense_in.min()) / (np.ptp(dense_in) + 1e-6)]
row = np.concatenate([cv2.resize((np.clip(t, 0, 1) * 255).astype(np.uint8), (384, 384)) for t in tiles], 1)
print(example.name, '| grey, limb-flattened, black top-hat, dense input')
display(Image(data=cv2.imencode('.png', row)[1].tobytes()))

## 3. Training (optional, `TRAIN = True`)
Training runs in three steps, in this order. The exact settings of every delivered model are in `train/provenance.json`.
1. **12 fold models.** Each model of fold *f* is trained on the other two folds of `train/folds.csv`. Image groups never cross folds.
   - `train/maskrcnn_ep24.py`: Mask R-CNN R50-FPN v2 (COCO init) for 24 epochs, then a ResNet18 U-Net crop refiner (ImageNet init) for 8 epochs. Grey input.
   - `train/maskrcnn_e034.py`: the same with the physics channels and 10 detector epochs.
   - `train/dense_unet.py`: ConvNeXt-T U-Net (ImageNet init) at 1536 for 30 epochs, seed 20261027 (`D1`) or 20261028 (`D2`).
2. **Collect.** `train/collect.py weights` copies the final checkpoints to `weights_retrained/` and writes `params_retrained.json`. The delivered `weights/` are not touched.
3. **C2 rescorer and threshold.** `train/fit_c2.py` runs the retrained fold models on their own held-out fold (out-of-fold, 8 views) and fuses the candidates. It then fits the 9-feature logistic rescorer and its threshold tau. `train/collect.py c2` stores both in `params_retrained.json`.

On one GPU the notebook adds `--nproc 0 --accum 2` to the Mask R-CNN commands (no DDP; the same effective batch as our 2 DDP ranks of batch 1). Costs we measured: Mask R-CNN about 2 h per fold on 2 x T4 (DDP, batch 1 per GPU); dense U-Net about 1 h per fold on an RTX 3080 Ti; step 3 about 1 h. The pretrained initial weights are downloaded once, so internet is needed for training only. Retrained models will not be bit-identical to the delivered ones (GPU type, library versions, DDP).

In [ ]:
PARAMS = 'params.json'
prov = json.loads((ROOT / 'train/provenance.json').read_text(encoding='utf-8'))
def train_args(m, one_gpu):
    args = [str(TRAIN_DATA) if a == '<data>' else a for a in m['args']]   # the path stays one argument (spaces, backslashes)
    if 'maskrcnn' in args[0]:
        args += ['--cap-seconds', '400000']                 # wall-clock guard only
        if one_gpu:
            args += ['--nproc', '0', '--accum', '2']        # one GPU, no DDP: same effective batch as our 2 DDP ranks of batch 1
    else:
        args = [a if a != '30000' else '400000' for a in args]
    return args
for m in prov['models']:
    print(m['id'], '|', m['command'])
if TRAIN:
    import torch
    one_gpu = torch.cuda.device_count() == 1
    for m in prov['models']:
        run(train_args(m, one_gpu))
    run([ROOT / 'train/collect.py', 'weights', '--runs', ROOT / 'train_runs'])
    run([ROOT / 'train/fit_c2.py', '--data', TRAIN_DATA, '--params', 'params_retrained.json', '--out', ROOT / 'c2_retrained.json'])
    run([ROOT / 'train/collect.py', 'c2', '--fit', ROOT / 'c2_retrained.json'])
    PARAMS = 'params_retrained.json'
print('inference uses', PARAMS)

## 4. Inference and post-processing
`code/e080_infer.py` is the same entry as the command line.
1. **Members.** 12 fold models x 8 views = 96 member predictions per image. Mask R-CNN detections are refined crop by crop. The dense probability maps become instances: probability > 0.5, 8-connected components, area >= 64 px.
2. **Fusion.** Instances of all members are clustered at IoU > 0.5. A cluster's fused score is the sum of its member confidences divided by the number of members; clusters below 0.30 are dropped, then NMS and overlap removal.
3. **Rescoring (C2).** Each fused instance gets 9 features: fused score, vote share, mean member score, share of families, share of views, log area, contrast, distance from the disk centre, log number of instances in the image. The saved logistic regression scores them, and instances with probability >= tau (0.29) are kept.
4. **Output.** One RLE mask per instance in the submission format. The CSV is validated at the end.

In [ ]:
cmd = [ROOT / 'code/e080_infer.py', '--images', TEST_IMAGES, '--out', OUT, '--params', PARAMS]
if LIMIT:
    cmd += ['--limit', str(LIMIT)]
run(cmd)
stats = json.loads(OUT.with_suffix('.stats.json').read_text(encoding='utf-8'))
print({k: stats[k] for k in ('valid', 'images', 'rows', 'seconds_per_image')})

## 5. Evaluation
The test labels are not public, so we evaluate out of fold on the training images. Each image is predicted by the four fold models that did not see its fold (32 members per image; the deployed ensemble uses 96). PQ follows the competition definition: IoU > 0.5 matches over every annotator record.

What "leave one fold out" covers here: C2 and tau are fitted on the out-of-fold candidates of two folds and applied to the third. The base models that produced those two folds' candidates were trained with the evaluated fold included, so this checks the C2 layer on fixed out-of-fold candidates, not a fully nested cross-validation of the whole pipeline.

`EVALUATE = True` recomputes this with `train/fit_c2.py --evaluate` for the models in use (about 1 GPU hour). Otherwise the stored result for the **delivered** weights (`docs/evaluation.json`) is shown, also after `TRAIN = True`. These are estimates on development data, which we reused many times, so they are optimistic. The public leaderboard score was 0.40.

In [ ]:
if EVALUATE:
    run([ROOT / 'train/fit_c2.py', '--data', TRAIN_DATA, '--params', PARAMS, '--out', ROOT / 'evaluation_run.json', '--evaluate'])
    ev = json.loads((ROOT / 'evaluation_run.json').read_text(encoding='utf-8'))
    print('evaluation of the models in use:', PARAMS)
else:
    ev = json.loads((ROOT / 'docs/evaluation.json').read_text(encoding='utf-8'))
    print('stored evaluation of the DELIVERED weights (params.json)' + (', not of the retrained models' if PARAMS != 'params.json' else ''))
for f, r in ev['evaluation']['leave_one_fold_out_pq'].items():
    print(f'fold {f}: PQ {r["pq"]:.4f} (tau {r["tau"]})')
print('pooled out-of-fold PQ', round(ev['evaluation']['pooled_pq'], 4))
print(f"C2 refitted on these candidates vs {ev.get('compared_with', 'params.json')}:", ev['vs_params_json'])

## 6. Look at the predictions
Each predicted instance outline is drawn on its H-alpha image.

In [ ]:
import csv
from collections import defaultdict
from pycocotools import mask as cm
csv.field_size_limit(10 ** 8)
per_image = defaultdict(list)
with open(OUT, newline='', encoding='utf-8') as fh:
    for row in csv.DictReader(fh):
        per_image[row['filament_id'].rsplit('_', 1)[0]].append(row['segmentation_rle'])
paths = {p.stem: p for p in TEST_IMAGES.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg')}
for pid, rles in list(per_image.items())[:3]:
    img = cv2.resize(cv2.imread(str(paths[pid])), (2048, 2048))
    for r in rles:
        m = cm.decode({'size': [2048, 2048], 'counts': r.encode('ascii')})
        cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(img, cs, -1, (0, 255, 255), 3)
    print(pid, len(rles), 'instances')
    display(Image(data=cv2.imencode('.jpg', cv2.resize(img, (768, 768)))[1].tobytes()))